# nano-dsv4.1f — prepare 3B pretraining tokens on Kaggle CPU

Enable Internet and use a CPU session. This builds **only FineWeb-Edu pretraining data** with your frozen nano tokenizer: approximately **3B nonpadding training tokens**, plus **10M held-out validation tokens**, packed into 8192-token rows. No mid-training, reasoning, agent, or SFT stages run.

Tokenization happens once on CPU. Training subsequently reads local token shards. The target includes BOS/EOS and excludes padding; each split can overshoot by less than one 8K segment. Expect roughly 6 GB of token IDs plus small packing metadata and padding. Actual real-token, LM-target, and physical-token totals are reported separately.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = 'https://github.com/xiayicheng3-code/nano-dsv4.1f.git'
REPO_REF = 'codex/pretrain-only-3b'
TOKENIZER_DATASET = 'xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb'
TRAIN_TOKENS = 3_000_000_000
VALIDATION_TOKENS = 10_000_000
SEQ_LEN = 8192
SEED = 1701
RUN_SMOKE = True
RUN_FULL = True

TEMP = Path('/kaggle/temp/nano-pretrain')
TEMP.mkdir(parents=True, exist_ok=True)
REPO_DIR = TEMP / 'repo'
DEPS = TEMP / 'deps'
PYTHON = Path(sys.executable)
OUTPUT = Path('/kaggle/working/nano-dsv41f-pretrain-3b-8k')
env = os.environ.copy()
env['TOKENIZERS_PARALLELISM'] = 'true'
env['RAYON_NUM_THREADS'] = str(len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1))
env['HF_HOME'] = str(TEMP / 'hf-cache')
env['HF_DATASETS_CACHE'] = str(TEMP / 'datasets-cache')
print({'threads': env['RAYON_NUM_THREADS'], 'train_tokens': TRAIN_TOKENS, 'validation_tokens': VALIDATION_TOKENS,
       'minimum_train_rows': (TRAIN_TOKENS + SEQ_LEN - 1) // SEQ_LEN})

In [ ]:
# Keep data-prep dependencies isolated without relying on Kaggle venv/ensurepip.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('The notebook checkout has local edits; preserve them before updating it.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DEPS.mkdir(parents=True, exist_ok=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-q', '--upgrade', '--target', str(DEPS),
                '-r', str(REPO_DIR / 'requirements-pretrain-data.txt')], check=True)
env['PYTHONPATH'] = str(DEPS) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
print('Repository commit:', COMMIT)
print('Data-prep dependencies:', DEPS)

In [ ]:
# Prefer the already-attached tokenizer dataset. Otherwise download the same dataset.
roots = [Path('/kaggle/input/datasets/xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb'),
         Path('/kaggle/input/nano-dsv41f-tokenizer-fineweb')]
candidates = sorted({p for root in roots if root.exists() for p in root.rglob('tokenizer.json')})
if not candidates:
    resolver = "import kagglehub, sys; print(kagglehub.dataset_download(sys.argv[1]))"
    result = subprocess.check_output([str(PYTHON), '-c', resolver, TOKENIZER_DATASET], text=True, env=env)
    tokenizer_root = Path(result.strip().splitlines()[-1])
    candidates = sorted(tokenizer_root.rglob('tokenizer.json'))
if len(candidates) != 1:
    raise RuntimeError(f'Expected one tokenizer.json in the selected dataset; found {candidates}')
TOKENIZER_PATH = candidates[0]
print('Tokenizer:', TOKENIZER_PATH)

SCRIPT = REPO_DIR / 'scripts/prepare_pretrain_corpus.py'
def prepare(output, train_tokens, validation_tokens):
    cmd = [str(PYTHON), '-u', str(SCRIPT),
           '--tokenizer', str(TOKENIZER_PATH), '--output-dir', str(output),
           '--train-tokens', str(train_tokens), '--validation-tokens', str(validation_tokens),
           '--seq-len', str(SEQ_LEN), '--seed', str(SEED)]
    result = subprocess.run(cmd, check=False, cwd=REPO_DIR, env=env)
    if result.returncode == 0:
        return
    manifest_path = Path(output) / 'manifest.json'
    complete = False
    if manifest_path.is_file():
        try:
            complete = bool(json.loads(manifest_path.read_text()).get('complete'))
        except (OSError, json.JSONDecodeError):
            pass
    # PyArrow/Python 3.12 can abort during interpreter shutdown after an early-stopped
    # parquet stream. Accept only SIGABRT after the builder already committed a complete manifest.
    if complete and result.returncode in (-6, 134):
        print(f'Ignoring shutdown-only SIGABRT after complete manifest: {output}')
        return
    raise subprocess.CalledProcessError(result.returncode, cmd)

In [ ]:
# A small live-data run catches access/schema/tokenizer errors before the full build.
if RUN_SMOKE:
    prepare(TEMP / 'smoke', 131_072, 16_384)

In [ ]:
# Rerunning keeps completed source files and retries only the interrupted source file.
# Never delete OUTPUT to retry. The builder rejects incompatible recipe changes.
if RUN_FULL:
    prepare(OUTPUT, TRAIN_TOKENS, VALIDATION_TOKENS)
    (OUTPUT / 'run_info.json').write_text(json.dumps({'repo_commit': COMMIT, 'repo_ref': REPO_REF,
        'tokenizer_dataset': TOKENIZER_DATASET}, indent=2) + '\n')

In [ ]:
if RUN_FULL:
    manifest = json.loads((OUTPUT / 'manifest.json').read_text())
    assert manifest['complete']
    for split in ('train', 'validation'):
        counts = manifest['splits'][split]
        print(split, counts)
        print('real-token utilization:', counts['real_tokens'] / counts['physical_tokens'])
    print('Total saved GB:', round(sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file()) / 1e9, 3))
    print('Output:', OUTPUT)
    reader_check = """
from scripts.prepare_pretrain_corpus import iter_pretrain_batches
import sys
batch = next(iter_pretrain_batches(sys.argv[1], batch_rows=1))
print({key: (value.shape, str(value.dtype)) for key, value in batch.items()})
"""
    subprocess.run([str(PYTHON), '-c', reader_check, str(OUTPUT)], check=True, cwd=REPO_DIR, env=env)

## Use the output

Save a Kaggle notebook version with outputs, then attach `nano-dsv41f-pretrain-3b-8k` to the TPU notebook. Keep the directory structure; compression is unnecessary.

`manifest.json` must have `complete: true`. The loader `scripts.prepare_pretrain_corpus.iter_pretrain_batches()` returns `input_ids`, `segment_ids`, and `token_mask` compatible with `put_training_batch()`. It never repeats the dataset implicitly. Training checkpoints must retain the data seed and consumed batch index.

Each completed input Parquet file is a restart checkpoint. A retry can reread/re-tokenize the current unfinished input file, but keeps all completed files. To resume in a different Kaggle session, copy the saved partial output into the same writable output path first; session-local files alone are not durable.

The 3B target is a data budget, independent of batch size and optimizer steps. Compute steps from the actual packed row count, global rows per microbatch, and gradient accumulation once the TPU stress test establishes a suitable batch. This notebook prepares data; it does not launch model training.